In [73]:
import pandas as pd
import matplotlib.pyplot as plt

In [74]:
train = pd.read_csv("train_final.csv")
test = pd.read_csv("test_final.csv")

In [75]:
import numpy as np
import pandas as pd


def build_features(
    train_df,
    test_df=None,
    lags=[1, 7, 14, 28],
    rolling_windows=[7, 14],
):
    """
    Builds forecasting features for train and optionally test.

    Parameters
    ----------
    train_df : pd.DataFrame
        Must contain:
        [
            "date",
            "store_id",
            "product_id",
            "price",
            "demand"
        ]

    test_df : pd.DataFrame or None
        Test dataframe WITHOUT demand.

    Returns
    -------
    train_feat : pd.DataFrame
    test_feat : pd.DataFrame or None
    """

    train_df = train_df.copy()

    # ---------------------------------------------------
    # DATE FEATURES
    # ---------------------------------------------------

    def add_date_features(df):

        df["date"] = pd.to_datetime(df["date"])

        df["day"] = df["date"].dt.day
        df["month"] = df["date"].dt.month
        df["year"] = df["date"].dt.year
        df["dayofweek"] = df["date"].dt.dayofweek
        df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

        df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

        return df

    train_df = add_date_features(train_df)

    # ---------------------------------------------------
    # SORT
    # ---------------------------------------------------

    group_cols = ["store_id", "product_id"]

    train_df = train_df.sort_values(group_cols + ["date"])

    # ---------------------------------------------------
    # PRICE FEATURES
    # ---------------------------------------------------

    train_df["price_change"] = (
        train_df.groupby(group_cols)["price"]
        .pct_change()
    )

    train_df["avg_price_product"] = (
        train_df.groupby("product_id")["price"]
        .transform("mean")
    )

    train_df["relative_price"] = (
        train_df["price"] / train_df["avg_price_product"]
    )

    # ---------------------------------------------------
    # HOLIDAY FEATURES
    # ---------------------------------------------------

    train_df["before_holiday"] = (
        train_df.groupby(group_cols)["is_holiday"]
        .shift(-1)
        .fillna(0)
    )

    train_df["after_holiday"] = (
        train_df.groupby(group_cols)["is_holiday"]
        .shift(1)
        .fillna(0)
    )

    # ---------------------------------------------------
    # LAG FEATURES
    # ---------------------------------------------------

    for lag in lags:

        train_df[f"lag_{lag}"] = (
            train_df.groupby(group_cols)["demand"]
            .shift(lag)
        )

    # ---------------------------------------------------
    # ROLLING FEATURES
    # ---------------------------------------------------

    for window in rolling_windows:

        train_df[f"rolling_mean_{window}"] = (
            train_df.groupby(group_cols)["demand"]
            .shift(1)
            .rolling(window)
            .mean()
        )

        train_df[f"rolling_std_{window}"] = (
            train_df.groupby(group_cols)["demand"]
            .shift(1)
            .rolling(window)
            .std()
        )

    # ---------------------------------------------------
    # TEST FEATURES
    # ---------------------------------------------------

    if test_df is None:
        return train_df, None

    test_df = test_df.copy()
    test_df = add_date_features(test_df)

    test_df = test_df.sort_values(group_cols + ["date"])

    # product average price from TRAIN ONLY
    avg_price_map = (
        train_df.groupby("product_id")["price"]
        .mean()
        .to_dict()
    )

    test_df["avg_price_product"] = (
        test_df["product_id"].map(avg_price_map)
    )

    test_df["relative_price"] = (
        test_df["price"] / test_df["avg_price_product"]
    )

    # ---------------------------------------------------
    # BUILD HISTORY
    # ---------------------------------------------------

    history = {}

    for _, row in train_df.iterrows():

        key = (row["store_id"], row["product_id"])

        history.setdefault(key, []).append(row["demand"])

    # ---------------------------------------------------
    # AUTOREGRESSIVE FEATURE CREATION
    # ---------------------------------------------------

    test_rows = []

    for _, row in test_df.iterrows():

        row = row.copy()

        key = (row["store_id"], row["product_id"])

        hist = history.get(key, [])

        # -------------------------
        # LAGS
        # -------------------------

        for lag in lags:

            if len(hist) >= lag:
                row[f"lag_{lag}"] = hist[-lag]
            else:
                row[f"lag_{lag}"] = np.nan

        # -------------------------
        # ROLLING
        # -------------------------

        for window in rolling_windows:

            if len(hist) >= window:

                vals = hist[-window:]

                row[f"rolling_mean_{window}"] = np.mean(vals)
                row[f"rolling_std_{window}"] = np.std(vals)

            else:

                row[f"rolling_mean_{window}"] = np.nan
                row[f"rolling_std_{window}"] = np.nan

        # -------------------------
        # PRICE FEATURES
        # -------------------------

        # approximate price_change
        # using last known train price

        last_train_rows = train_df[
            (train_df["store_id"] == key[0]) &
            (train_df["product_id"] == key[1])
        ]

        if len(last_train_rows) > 0:

            last_price = last_train_rows.iloc[-1]["price"]

            row["price_change"] = (
                (row["price"] - last_price) / last_price
            )

        else:
            row["price_change"] = np.nan

        # -------------------------
        # HOLIDAY FEATURES
        # -------------------------

        row["before_holiday"] = 0
        row["after_holiday"] = 0

        test_rows.append(row)

    test_feat = pd.DataFrame(test_rows)

    return train_df, test_feat

In [76]:
train_feat, test_feat = build_features(train, test)

In [83]:
test_feat

,row_id,date,store_id,city,region,product_id,product_name,category,price,loyalty_day,...,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,price_change,before_holiday,after_holiday
0,0,2025-05-11,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,1,...,4,5,5,7.142857,1.641304,6.071429,2.491823,0.0,0,0
580,580,2025-05-12,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,0,...,4,5,5,7.142857,1.641304,6.071429,2.491823,0.0,0,0
1160,1160,2025-05-13,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,1,...,4,5,5,7.142857,1.641304,6.071429,2.491823,0.0,0,0
1740,1740,2025-05-14,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,1,...,4,5,5,7.142857,1.641304,6.071429,2.491823,0.0,0,0
2320,2320,2025-05-15,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,1,...,4,5,5,7.142857,1.641304,6.071429,2.491823,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5799,5799,2025-05-20,CM-TIM-01,Timișoara,Banat,P0058,Cremă de mâini 75ml,Personal Care & Cleaning,15.11,0,...,2,0,0,1.000000,0.925820,0.642857,0.894997,0.0,0,0
6379,6379,2025-05-21,CM-TIM-01,Timișoara,Banat,P0058,Cremă de mâini 75ml,Personal Care & Cleaning,15.11,0,...,2,0,0,1.000000,0.925820,0.642857,0.894997,0.0,0,0
6959,6959,2025-05-22,CM-TIM-01,Timișoara,Banat,P0058,Cremă de mâini 75ml,Personal Care & Cleaning,15.11,0,...,2,0,0,1.000000,0.925820,0.642857,0.894997,0.0,0,0
7539,7539,2025-05-23,CM-TIM-01,Timișoara,Banat,P0058,Cremă de mâini 75ml,Personal Care & Cleaning,15.11,0,...,2,0,0,1.000000,0.925820,0.642857,0.894997,0.0,0,0


In [86]:
cat_features = [
    "store_id",
    "city",
    "region",
    "product_id",
    "product_name",
    "category",
    "holiday_name"
]

In [ ]:
from catboost import CatBoostRegressor
from sklearn.svm import SVR

model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

In [92]:
TARGET = "demand"

DROP_COLS = [
    "row_ID",
    "date",
    "demand"
]

FEATURES = [
    c for c in train_feat.columns
    if c not in DROP_COLS
]

In [98]:
X_train = train_feat[FEATURES]
y_train = train_feat["demand"]
X_test = test_feat[FEATURES]

In [99]:
cat_features = [
    "store_id",
    "city",
    "region",
    "product_id",
    "product_name",
    "category",
    "holiday_name"
]

for c in cat_features:

    X_train[c] = (
        X_train[c]
        .fillna("UNKNOWN")
        .astype(str)
    )

    X_test[c] = (
        X_test[c]
        .fillna("UNKNOWN")
        .astype(str)
    )

In [100]:
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    use_best_model=True
)

You should provide test set for use best model. use_best_model parameter has been switched to false value.


0:	learn: 2.7255765	total: 105ms	remaining: 5m 16s
100:	learn: 1.8819597	total: 9.89s	remaining: 4m 43s
200:	learn: 1.8406451	total: 19.6s	remaining: 4m 32s
300:	learn: 1.8201701	total: 29.4s	remaining: 4m 24s
400:	learn: 1.8047812	total: 39.2s	remaining: 4m 13s
500:	learn: 1.7911910	total: 49.5s	remaining: 4m 7s
600:	learn: 1.7789651	total: 59.8s	remaining: 3m 58s
700:	learn: 1.7688325	total: 1m 10s	remaining: 3m 50s
800:	learn: 1.7596073	total: 1m 20s	remaining: 3m 41s
900:	learn: 1.7505605	total: 1m 31s	remaining: 3m 32s
1000:	learn: 1.7416794	total: 1m 42s	remaining: 3m 24s
1100:	learn: 1.7334299	total: 1m 52s	remaining: 3m 14s
1200:	learn: 1.7256953	total: 2m 3s	remaining: 3m 5s
1300:	learn: 1.7190404	total: 2m 14s	remaining: 2m 55s
1400:	learn: 1.7133647	total: 2m 24s	remaining: 2m 44s
1500:	learn: 1.7074545	total: 2m 35s	remaining: 2m 34s
1600:	learn: 1.7011563	total: 2m 45s	remaining: 2m 24s
1700:	learn: 1.6952364	total: 2m 55s	remaining: 2m 14s
1800:	learn: 1.6902341	total: 3m

CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=3000, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=100)

In [ ]:
# EXACT SAME FEATURE ORDER USED DURING TRAINING

X_test = test_feat[X_train.columns]

,row_id,store_id,city,region,product_id,product_name,category,price,loyalty_day,is_holiday,day,month,year,dayofweek,weekofyear,is_weekend
0,0,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,1,0,11,5,2025,6,19,1
1,1,CM-BAC-01,Bacău,Moldova,P0002,Apă minerală carbogazoasă 1.5L,Beverages,8.68,1,0,11,5,2025,6,19,1
2,2,CM-BAC-01,Bacău,Moldova,P0003,Cola 2L,Beverages,9.73,1,0,11,5,2025,6,19,1
3,3,CM-BAC-01,Bacău,Moldova,P0004,Suc natural portocale 1L,Beverages,8.60,1,0,11,5,2025,6,19,1
4,4,CM-BAC-01,Bacău,Moldova,P0005,Bere lager 0.5L,Beverages,4.26,1,0,11,5,2025,6,19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8115,8115,CM-TIM-01,Timișoara,Banat,P0054,Mălai 1kg,Pantry,14.63,0,0,24,5,2025,5,21,1
8116,8116,CM-TIM-01,Timișoara,Banat,P0055,Joc puzzle mini,Toys & Games,10.73,0,0,24,5,2025,5,21,1
8117,8117,CM-TIM-01,Timișoara,Banat,P0056,Bandă elastică fitness,Sports & Hobbies,55.49,0,0,24,5,2025,5,21,1
8118,8118,CM-TIM-01,Timișoara,Banat,P0057,Lingură silicon 30cm,Home & Kitchen,9.86,0,0,24,5,2025,5,21,1


In [103]:
pred = model.predict(X_test)

In [104]:
ids = test["row_id"]

In [105]:
f = open("ans.csv", 'w')

f.write("row_id,demand\n")

for i, p in enumerate(pred):
    f.write(str(ids[i]) + "," + str(p) + '\n')
f.close()

In [ ]:
train["category"].unique()

<ArrowStringArray>
[               'Beverages',                   'Pantry',
   'Snacks & Confectionery',     'Frozen & Ready Meals',
         'Sports & Hobbies',             'Toys & Games',
           'Home & Kitchen', 'Personal Care & Cleaning']
Length: 8, dtype: str

In [ ]:
train[train["is_weekend"] == 1]

,row_id,store_id,city,region,product_id,product_name,category,price,loyalty_day,is_holiday,demand,day,month,year,dayofweek,weekofyear,is_weekend
0,725,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.08,0,0,7,10,5,2025,5,19,1
1,1451,CM-BAC-01,Bacău,Moldova,P0002,Apă minerală carbogazoasă 1.5L,Beverages,8.68,0,0,8,10,5,2025,5,19,1
2,2177,CM-BAC-01,Bacău,Moldova,P0003,Cola 2L,Beverages,9.73,0,0,2,10,5,2025,5,19,1
3,2903,CM-BAC-01,Bacău,Moldova,P0004,Suc natural portocale 1L,Beverages,8.60,0,0,0,10,5,2025,5,19,1
4,3629,CM-BAC-01,Bacău,Moldova,P0005,Bere lager 0.5L,Beverages,4.26,0,0,0,10,5,2025,5,19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
418175,417455,CM-TIM-01,Timișoara,Banat,P0054,Mălai 1kg,Pantry,NaN,0,0,0,20,5,2023,5,20,1
418176,418181,CM-TIM-01,Timișoara,Banat,P0055,Joc puzzle mini,Toys & Games,NaN,0,0,0,20,5,2023,5,20,1
418177,418907,CM-TIM-01,Timișoara,Banat,P0056,Bandă elastică fitness,Sports & Hobbies,NaN,0,0,0,20,5,2023,5,20,1
418178,419633,CM-TIM-01,Timișoara,Banat,P0057,Lingură silicon 30cm,Home & Kitchen,NaN,0,0,0,20,5,2023,5,20,1
